# Cross-Country Solar Potential Comparison
**MoonLight Energy Solutions | Task 3**

Branch: `compare-countries`

**Objective:** Synthesise cleaned datasets from Benin, Sierra Leone, and Togo
to identify relative solar potential and key differences.

**References:**
- [scipy.stats.f_oneway](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.f_oneway.html)
- [scipy.stats.kruskal](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kruskal.html)
- Montgomery, D.C. (2017). *Design and Analysis of Experiments*. Wiley.

## 1. Imports & Load Cleaned Data

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False})

COUNTRY_FILES = {
    "Benin":        "E:/project 2026/moonlight-solar/data/benin_clean.csv",
    "Sierra Leone": "E:/project 2026/moonlight-solar/data/sierraleone_clean.csv",
    "Togo":         "E:/project 2026/moonlight-solar/data/togo_clean.csv",
}
COLORS = {
    "Benin":        "#E63946",
    "Sierra Leone": "#457B9D",
    "Togo":         "#2A9D8F",
}
IRRAD_COLS = ["GHI", "DNI", "DHI"]

In [ ]:
frames = {}
for country, path in COUNTRY_FILES.items():
    try:
        df = pd.read_csv(path)
        if "Timestamp" in df.columns:
            df["Timestamp"] = pd.to_datetime(df["Timestamp"])
            df = df.set_index("Timestamp").sort_index()
        df["Country"] = country
        frames[country] = df
        print(f"✓ {country:15s}  {len(df):>7,} rows  "
              f"{df.index.min().date()} → {df.index.max().date()}")
    except FileNotFoundError:
        print(f"✗ {country}: file not found at {path}")

print(f"\nLoaded {len(frames)}/3 countries.")

## 2. Boxplot Comparisons — GHI, DNI, DHI

In [ ]:
combined = pd.concat(frames.values(), ignore_index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, col in zip(axes, IRRAD_COLS):
    data   = [frames[c][col].dropna() for c in frames]
    labels = list(frames.keys())
    bp = ax.boxplot(data, patch_artist=True, medianprops=dict(color="white", lw=2),
                    whiskerprops=dict(lw=1.2), capprops=dict(lw=1.2))
    for patch, country in zip(bp["boxes"], labels):
        patch.set_facecolor(COLORS[country])
        patch.set_alpha(0.85)
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylabel(f"{col} (W/m²)")
    ax.set_title(f"{col} by Country", fontweight="bold")

fig.suptitle("Solar Irradiance Distribution by Country",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Summary Statistics Table

In [ ]:
rows = []
for country, df in frames.items():
    for col in IRRAD_COLS:
        if col in df.columns:
            rows.append({
                "Country": country,
                "Metric":  col,
                "Mean":    df[col].mean(),
                "Median":  df[col].median(),
                "Std Dev": df[col].std(),
                "Max":     df[col].max(),
                "Min":     df[col].min(),
            })

summary = pd.DataFrame(rows).round(2)
display(summary)

## 4. Statistical Testing

We apply two tests:
1. **One-way ANOVA** — parametric; assumes roughly normal distributions.
2. **Kruskal-Wallis** — non-parametric alternative; robust to non-normality.

**Hypotheses (GHI example):**
- H₀: Mean GHI is equal across all three countries.
- H₁: At least one country has a different mean GHI.

**Significance level:** α = 0.05

In [ ]:
for col in IRRAD_COLS:
    groups = [frames[c][col].dropna().values for c in frames if col in frames[c].columns]
    f_stat, p_anova   = stats.f_oneway(*groups)
    h_stat, p_kruskal = stats.kruskal(*groups)
    print(f"── {col} ──")
    print(f"  ANOVA:          F = {f_stat:>10.2f},  p = {p_anova:.2e}  "
          f"{'✓ significant' if p_anova < 0.05 else '✗ not significant'}")
    print(f"  Kruskal-Wallis: H = {h_stat:>10.2f},  p = {p_kruskal:.2e}  "
          f"{'✓ significant' if p_kruskal < 0.05 else '✗ not significant'}")
    print()

## 5. Country Ranking Bar Chart (Average GHI)

In [ ]:
avg_ghi = {c: df["GHI"].mean() for c, df in frames.items() if "GHI" in df.columns}
avg_ghi = dict(sorted(avg_ghi.items(), key=lambda x: x[1], reverse=True))

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(avg_ghi.keys(), avg_ghi.values(),
              color=[COLORS[c] for c in avg_ghi], alpha=0.88, edgecolor="white")
for bar, val in zip(bars, avg_ghi.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f"{val:.1f}", ha="center", va="bottom", fontweight="bold", fontsize=11)
ax.set_ylabel("Average GHI (W/m²)")
ax.set_title("Country Ranking by Average GHI", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Key Observations

- 🔹 **Observation 1 — Highest Solar Resource:** Benin (Malanville) records the
  highest median GHI, reflecting its Sahelian location with drier, clearer skies
  and minimal cloud cover year-round.

- 🔹 **Observation 2 — Variability Risk:** Sierra Leone (Bumbuna) shows the greatest
  GHI variability (widest IQR and standard deviation), driven by its humid tropical
  climate and pronounced wet season. This increases storage requirements and
  intermittency risk.

- 🔹 **Observation 3 — Balanced Profile:** Togo (Dapaong) offers competitive median
  GHI with lower variability than Sierra Leone, making it the most stable option
  for utility-scale projects requiring firm capacity commitments.

---
*End of cross-country comparison notebook.*